# 第二讲交互实验：Tensor、FLOPs 与训练显存

本 Notebook 通过可执行的小实验，帮助你把张量形状、dtype、矩阵计算、反向传播和显存账本联系起来。理论推导见 [第二讲统一讲义](../../lectures/02_pytorch_resource_accounting.md)。

## 环境与执行约定

需要 Python、PyTorch 和 einops。尚未安装时，可在终端执行：

`python -m pip install jupyter torch einops`

建议从仓库根目录启动 Jupyter。先预测输出，再运行 Cell；允许修改 shape、dtype、batch size 等参数观察变化。Notebook 不包含预制运行结果，请在本地执行。

In [ ]:
import copy
from pathlib import Path
import sys
import torch
from einops import einsum, rearrange, reduce

print("PyTorch 版本：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())
print("当前默认设备：", "cuda" if torch.cuda.is_available() else "cpu")

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (path for path in candidates if (path / "code/lecture_02/resource_accounting_demo.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("没有找到仓库根目录；请从仓库中启动 Jupyter。")
print("仓库根目录：", ROOT)

## 实验 1：从 Token IDs 到 Embedding

形状中的每个轴都应有明确语义。这里使用 `B=2`、`T=4`、`D=3`，观察 Embedding 如何增加特征维度，而不是增加 token 位置数。

In [ ]:
tokens = torch.tensor([[12, 35, 9, 8], [2, 15, 7, 6]])
embedding = torch.nn.Embedding(num_embeddings=256, embedding_dim=3)
x = embedding(tokens)

print("Token IDs shape:", tuple(tokens.shape))
print("Embedding shape:", tuple(x.shape))
print("Embedding 前两个向量：\n", x[0, :2])

assert tuple(tokens.shape) == (2, 4)
assert tuple(x.shape) == (2, 4, 3)

## 实验 2：dtype 如何改变张量字节数？

逻辑数据量可以按 `numel × element_size` 计算。它不等于训练进程的完整显存占用，因为还可能有梯度、激活、优化器状态和临时工作空间。

In [ ]:
def tensor_nbytes(tensor):
    return tensor.numel() * tensor.element_size()

for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    tensor = torch.zeros((2, 4, 3), dtype=dtype)
    print(
        f"dtype={dtype}, 元素数={tensor.numel()}, "
        f"bytes/元素={tensor.element_size()}, "
        f"逻辑数据量={tensor_nbytes(tensor)} bytes"
    )

## 实验 3：用 einops 显式表达轴

`einsum` 会对输入中存在、但输出表达式中省略的轴求和；`reduce` 对指定轴执行归约；`rearrange` 负责拆分、合并或重排轴。先判断每个输出 shape，再运行。

In [ ]:
x = torch.ones(2, 3, 4)  # batch, seq, hidden
w = torch.ones(4, 5)     # hidden, out

y = einsum(x, w, "batch seq hidden, hidden out -> batch seq out")
pooled = reduce(y, "batch seq out -> batch out", "mean")
heads = rearrange(
    x, "batch seq (heads dim) -> batch heads seq dim", heads=2
)

print("einsum:", tuple(y.shape))
print("reduce:", tuple(pooled.shape))
print("rearrange:", tuple(heads.shape))

assert tuple(y.shape) == (2, 3, 5)
assert tuple(pooled.shape) == (2, 5)
assert tuple(heads.shape) == (2, 2, 3, 2)
assert torch.equal(y, x @ w)

## 实验 4：亲手核算矩阵乘法 FLOPs

对于 `(M, K) @ (K, N)`，输出共有 `M×N` 个元素。每个元素有 `K` 次乘法与 `K−1` 次加法，因此精确的基本算术操作计数为 `MN(2K−1)`；常用近似为 `2MKN`。

In [ ]:
def matmul_flops_exact(m, k, n):
    if min(m, k, n) <= 0:
        raise ValueError("矩阵维度必须为正数")
    return m * n * (2 * k - 1)

def matmul_flops_approx(m, k, n):
    if min(m, k, n) <= 0:
        raise ValueError("矩阵维度必须为正数")
    return 2 * m * k * n

M, K, N = 2, 3, 4
print("输出元素数：", M * N)
print("精确基本操作计数：", matmul_flops_exact(M, K, N))
print("近似 FLOPs：", matmul_flops_approx(M, K, N))

assert matmul_flops_exact(2, 3, 4) == 40
assert matmul_flops_approx(2, 3, 4) == 48

## 实验 5：算术强度与理想时间下界

给定总计算量 `F`、搬运字节数 `M_bytes`、峰值算力 `P` 与内存带宽 `W`：

- 算术强度：`I = F / M_bytes`
- 理想时间下界：`t ≥ max(F/P, M_bytes/W)`
- Roofline 吞吐上界：`R ≤ min(P, I×W)`

下面的数值是**假设输入**，不是某张 GPU 的测量结果。

In [ ]:
F = 1e12          # FLOPs
M_bytes = 1e10    # bytes
peak_flops = 100e12  # FLOP/s
bandwidth = 2e12     # bytes/s

intensity = F / M_bytes
compute_bound_s = F / peak_flops
memory_bound_s = M_bytes / bandwidth
time_lower_bound_s = max(compute_bound_s, memory_bound_s)
ridge_point = peak_flops / bandwidth
roofline_throughput = min(peak_flops, intensity * bandwidth)

print(f"算术强度：{intensity:.1f} FLOPs/byte")
print(f"Roofline 转折点：{ridge_point:.1f} FLOPs/byte")
print(f"计算时间下界：{compute_bound_s:.4f} s")
print(f"搬运时间下界：{memory_bound_s:.4f} s")
print(f"理想总时间下界：{time_lower_bound_s:.4f} s")
print(f"理想吞吐上界：{roofline_throughput:.3e} FLOP/s")

## 实验 6：反向传播实际产生什么梯度？

对 `Y = XW`，反向传播不仅需要前向结果，还会计算输入梯度和权重梯度。下面使用一个小型张量，直接观察参数的 `.grad`。

In [ ]:
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
w = torch.ones((3, 2), requires_grad=True)

y = x @ w
loss = y.sum()
loss.backward()

print("Y shape:", tuple(y.shape))
print("loss:", loss.item())
print("W.grad shape:", tuple(w.grad.shape))
print("W.grad:\n", w.grad)

assert tuple(w.grad.shape) == (3, 2)
assert torch.equal(w.grad, torch.tensor([[5., 5.], [7., 7.], [9., 9.]]))

## 实验 7：参数、梯度与 Adam 状态是不同对象

观察一次标量训练步骤。优化器状态通常在第一次 `step()` 时建立；下面打印当前梯度、更新后的参数，以及 Adam 的一阶、二阶状态。

In [ ]:
w = torch.nn.Parameter(torch.tensor(3.0))
optimizer = torch.optim.AdamW([w], lr=0.1)

x = torch.tensor(2.0)
loss = 0.5 * (w * x - 4.0).square()
loss.backward()

gradient_before_step = w.grad.item()
optimizer.step()
state = optimizer.state[w]

print("更新前梯度：", gradient_before_step)
print("更新后参数：", w.item())
print("Adam exp_avg：", state["exp_avg"].item())
print("Adam exp_avg_sq：", state["exp_avg_sq"].item())

## 实验 8：激活张量随 micro-batch 大小怎样变化？

只计算一个假设激活张量的逻辑大小：shape 为 `(B, T, D)`，每元素 2 bytes。它不是完整训练显存预测；这里只比较 batch 维的影响。

In [ ]:
T, D, bytes_per_element = 128, 768, 2

for batch_size in [8, 4, 2, 1]:
    elements = batch_size * T * D
    mib = elements * bytes_per_element / (1024 ** 2)
    print(
        f"B={batch_size}: 元素数={elements:,}, "
        f"单个 BF16 激活张量={mib:.3f} MiB"
    )

## 实验 9：验证梯度累积与一次性 batch 的梯度

目标是验证：当 micro-batch 大小相同、损失按 micro-batch 数正确缩放，并且只在最后更新参数时，累积梯度应与一次性处理相同 batch 的梯度一致（浮点误差范围内）。

先预测最后的 `torch.allclose` 是否为 `True`。

In [ ]:
torch.manual_seed(0)
x_full = torch.randn(8, 3)
y_full = torch.randn(8, 1)

full_batch_model = torch.nn.Linear(3, 1, bias=False)
micro_batch_model = copy.deepcopy(full_batch_model)
loss_fn = torch.nn.MSELoss(reduction="mean")

# 方式 A：一次处理完整 batch
full_batch_model.zero_grad()
full_loss = loss_fn(full_batch_model(x_full), y_full)
full_loss.backward()
full_grad = full_batch_model.weight.grad.detach().clone()

# 方式 B：4 个等大小 micro-batch，累积梯度后再更新
micro_batch_model.zero_grad()
for x_micro, y_micro in zip(x_full.chunk(4), y_full.chunk(4)):
    micro_loss = loss_fn(micro_batch_model(x_micro), y_micro) / 4
    micro_loss.backward()
micro_grad = micro_batch_model.weight.grad.detach().clone()

print("完整 batch loss：", full_loss.item())
print("完整 batch 梯度：\n", full_grad)
print("累积梯度：\n", micro_grad)
print("梯度是否一致：", torch.allclose(full_grad, micro_grad, atol=1e-6))

assert torch.allclose(full_grad, micro_grad, atol=1e-6)

## 复盘与下一步

1. 为什么 Tensor 的逻辑数据量不等于训练任务的峰值显存？
2. `einsum` 中输入有、输出没有的轴意味着什么？
3. 为什么 FLOPs 多不一定耗时更长？还需要哪些硬件信息？
4. 梯度累积减少了什么，又不会自动减少什么？
5. 激活检查点为何是用计算换内存？

参考：[第二讲统一讲义](../../lectures/02_pytorch_resource_accounting.md) · [资源核算脚本](../../code/lecture_02/resource_accounting_demo.py) · [训练显存脚本](../../code/lecture_02/training_memory_walkthrough.py) · [单元测试](../../tests/lecture_02/test_resource_accounting_demo.py)